# Bronze Layer
Tạo tầng Bronze cho Lakehouse: ingest dữ liệu CSV Olist vào Delta Lake.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import *

# 1. Khởi tạo Spark với cấu hình Lakehouse
builder = SparkSession.builder.appName("OlistBronzeStreaming") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 2. Định nghĩa cấu trúc thư mục
raw_root = "/home/jovyan/work/data"
bronze_root = "/home/jovyan/work/delta_lake/bronze"
checkpoint_root = "/home/jovyan/work/delta_lake/_checkpoints"

# Danh sách bảng cần demo
datasets = {
    "customers": "olist_customers_dataset",
    "orders": "olist_orders_dataset",
    "order_items": "olist_order_items_dataset",
    "products": "olist_products_dataset",
    "translations": "product_category_name_translation"
}

streaming_queries = {}

for table_name, file_prefix in datasets.items():
    dst_path = f"{bronze_root}/{table_name}"
    cp_path = f"{checkpoint_root}/{table_name}"
    
    print(f"--- Đang thiết lập Ingestion Stream cho: {table_name} ---")
    
    # 3. Lấy Schema từ file mẫu đầu tiên để định nghĩa cấu trúc Stream
    # Quy cách: Streaming luôn cần Schema cố định để đảm bảo Data Integrity
    sample_path = f"{raw_root}/{file_prefix}.csv"
    base_schema = spark.read.option("header", True).option("inferSchema", True).csv(sample_path).schema

    # 4. Thiết lập luồng Stream theo dõi thư mục
    # Sử dụng Wildcard (*) để nhận diện file gốc và các file bổ sung (part2, part3...)
    df_stream = spark.readStream \
            .option("header", True) \
            .option("pathGlobFilter", f"{file_prefix}*.csv") \
            .option("multiLine", True) \
            .option("ignoreLeadingWhiteSpace", True) \
            .option("ignoreTrailingWhiteSpace", True) \
            .schema(base_schema) \
            .csv(raw_root)

    # 5. Thêm Metadata để phục vụ Lineage và Audit (Rất quan trọng trong Bronze Layer)
    df_bronze = df_stream.withColumn("ingested_at", F.current_timestamp()) \
                         .withColumn("source_file", F.input_file_name())

    # 6. Ghi xuống Delta Lake (Append Mode)
    # Checkpoint giúp Spark không bao giờ nạp trùng dữ liệu đã xử lý
    query = df_bronze.writeStream \
        .format("delta") \
        .outputMode("append") \
        .option("checkpointLocation", cp_path) \
        .trigger(processingTime='5 seconds') \
        .start(dst_path)
    
    streaming_queries[table_name] = query

print(">>> HỆ THỐNG INGESTION ĐANG SẴN SÀNG.")

--- Đang thiết lập Ingestion Stream cho: customers ---
--- Đang thiết lập Ingestion Stream cho: orders ---
--- Đang thiết lập Ingestion Stream cho: order_items ---
--- Đang thiết lập Ingestion Stream cho: products ---
--- Đang thiết lập Ingestion Stream cho: translations ---
>>> HỆ THỐNG INGESTION ĐANG SẴN SÀNG.


In [2]:
spark.streams.active

In [3]:
import time
from pyspark.sql.utils import AnalysisException

path_customers = "/home/jovyan/work/delta_lake/bronze/customers"

print("Đang kiểm tra dữ liệu...")

# Thử lại tối đa 5 lần, mỗi lần cách nhau 3 giây nếu chưa thấy folder
for i in range(5):
    try:
        count = spark.read.format("delta").load(path_customers).count()
        print(f"Thành công! Số dòng hiện tại trong Bronze Customers: {count:,}")
        break
    except AnalysisException:
        print(f"Lần thử {i+1}: Dữ liệu đang được khởi tạo, vui lòng đợi...")
        time.sleep(3)
else:
    print("Lỗi: Sau 15 giây vẫn chưa tìm thấy bảng Delta. Hãy kiểm tra lại Cell 1 xem có báo lỗi gì không.")

Đang kiểm tra dữ liệu...
Lần thử 1: Dữ liệu đang được khởi tạo, vui lòng đợi...
Thành công! Số dòng hiện tại trong Bronze Customers: 99,441


In [4]:
#dừng tất cả các luồng tránh treo máy
for table, query in streaming_queries.items():
    print(f"Đang dừng luồng: {table}...")
    query.stop()

Đang dừng luồng: customers...
Đang dừng luồng: orders...
Đang dừng luồng: order_items...
Đang dừng luồng: products...
Đang dừng luồng: translations...
